In [ ]:
# Source - https://stackoverflow.com/a
# Posted by dfrankow
# Retrieved 2026-01-16, License - CC BY-SA 3.0

options(repr.matrix.max.rows=600, repr.matrix.max.cols=200)

In [71]:
# %%
import logging
import os

import polars as pl
import pandas as pd



# %%
def read_gtdbtk(output_directory, taxonomy_only=True, remove_empty_ranks=False):
    if not taxonomy_only:
        raise NotImplementedError("Only taxonomy is supported for now")
    taxonomies = {}
    bac_taxonomy_file = os.path.join(output_directory, 'gtdbtk.bac120.summary.tsv')
    logging.debug('Reading taxonomy from %s' % bac_taxonomy_file)
    d = pl.read_csv(bac_taxonomy_file, separator='\t')
    if remove_empty_ranks:
        empty_ranks = ['d__', 'p__', 'c__', 'o__', 'f__', 'g__', 's__']
    for row in d.rows(named=True):
        tax = row['classification']
        if remove_empty_ranks:
            tax = ';'.join([x for x in tax.split(';') if x.strip() not in empty_ranks])
        taxonomies[row['user_genome']] = tax
    logging.debug("Read %d taxonomies from Bacteria" % len(taxonomies))

    # Archaea
    arc_taxonomy_file = os.path.join(output_directory, 'gtdbtk.ar53.summary.tsv')
    logging.debug('Reading taxonomy from %s' % arc_taxonomy_file)
    d = pl.read_csv(arc_taxonomy_file, separator='\t')
    num_archaea = 0
    for row in d.rows(named=True):
        tax = row['classification']
        if remove_empty_ranks:
            tax = ';'.join([x for x in tax.split(';') if x.strip() not in empty_ranks])
        taxonomies[row['user_genome']] = tax
        num_archaea += 1
    logging.debug("Read %d new archaeal taxonomies, so %d total" % (num_archaea, len(taxonomies)))
    return taxonomies


In [5]:
class Args:
    known_genome_list = '../1_novel_strains/shadow_genome_paths.csv'
    novel_genome_gtdbtk_output = '../4_complex_and_novel/gtdbtk_batchfile.random1000.gtdbtk_r207'
    novel_genome_list = '../4_complex_and_novel/gtdbtk_batchfile.random1000.csv'
    gtdb_bac_metadata = '../bac120_metadata_r207.tsv'
    gtdb_ar_metadata = "../ar53_metadata_r207.tsv"
args = Args()

In [6]:
genomes = (
    pl.read_csv(args.known_genome_list,
                separator = '\t', has_header = False,
                new_columns = ['genome', 'path'])
        .with_columns(
            # Make paths relative to input file
            pl.col('path').map_elements(
                lambda x: os.path.normpath(os.path.join(
                    os.getcwd(),
                    os.path.dirname(args.known_genome_list),
                    x
                )),
                return_dtype = pl.String()
            )
        )
)
logging.info(f"Read {len(genomes)} genome fasta paths.")

In [ ]:
bac = pl.read_csv(args.gtdb_bac_metadata, separator = '\t',
                    infer_schema_length = 100000,
                    ignore_errors = True)
ar = pl.read_csv(args.gtdb_ar_metadata, separator = '\t',
                    infer_schema_length = 100000,
                    ignore_errors = True)
metadata = pl.concat([
    bac.select('accession', 'genome_size', 'gtdb_taxonomy'),
    ar.select('accession', 'genome_size', 'gtdb_taxonomy'),
])
logging.info(f"Read {len(metadata)} GTDB metadata entries.")

# get rid of GB_, RS_
metadata = metadata.with_columns(pl.col('accession').str.slice(3).alias('genome'))

In [10]:
# Shuffle genomes order so we get randomness
#    metadata = metadata.sample(fraction=1)

known_info = (
    genomes.join(metadata, on = 'genome', how = 'inner')
        .select('path', 'genome', pl.col('gtdb_taxonomy').alias('taxonomy'))
)

r207_taxonomy = read_gtdbtk(args.novel_genome_gtdbtk_output, remove_empty_ranks=True)

In [11]:
# Read list of genome paths
novel_genome_list = (
    pl.read_csv(args.novel_genome_list, separator = '\t',
                has_header = False,
                new_columns = ['path', 'genome'])
        .with_columns(
            # Make paths relative to input file
            pl.col('path').map_elements(
                lambda x: os.path.normpath(os.path.join(
                    os.getcwd(),
                    os.path.dirname(args.novel_genome_list),
                    x
                )),
                return_dtype = pl.String()
            )
        )
)

In [12]:
# Merge with GTDBTK output
novel_info = novel_genome_list.with_columns(
    pl.col('genome').replace(r207_taxonomy).alias('taxonomy')
).with_columns(
    pl.col('taxonomy')
    .str.extract("(^d|;[pcofgs])__[^;]+$").alias("known_at")
).with_columns(
    pl.col('known_at')
    .replace_strict({";s": 0, ";g": 1, ";f": 2, ";o": 3, ";c": 4, ";p": 5, "d": 6})
    .alias('known_at')
).filter(pl.col('known_at') > 0)

In [43]:
all_info = pl.concat(
    [known_info,
     novel_info.drop("known_at")
    ]
).with_columns(n = 1)
all_info.get_column("taxonomy").to_list()[0]

'd__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Enterobacterales;f__Vibrionaceae;g__Vibrio;s__Vibrio coralliilyticus'

In [61]:
sp_info = (
    all_info
    .filter(pl.col('taxonomy').str.contains(";s__"))
    .group_by(pl.col('taxonomy').str.replace("^.+;[^g]__[^;]+;", ""))
    .agg(pl.col('n').sum())
    .sort(pl.col("taxonomy"))
)
sp_info.get_column("taxonomy").to_list()[:5]

['g__14-2;s__14-2 sp000403255',
 'g__14-2;s__14-2 sp000403315',
 'g__14-2;s__14-2 sp910576385',
 'g__2-02-FULL-39-32;s__2-02-FULL-39-32 sp001800135',
 'g__2-02-FULL-42-43;s__2-02-FULL-42-43 sp001801425']

In [85]:
gn_info = (
    all_info
    .filter(pl.col('taxonomy').str.contains(";g__"))
    .group_by(pl.col('taxonomy').str.replace("^([^f]__[^;]+;)+", "").str.replace(";s__.+$", ""))
    .agg(pl.col('n').sum())
    .filter(pl.col('n') > 1)
    .sort(pl.col('n'), pl.col("taxonomy"), descending = [True, False])
    .head(2)
    .with_columns(pl.lit("g").alias("rank"))
)
gn_info.get_column("taxonomy").to_list()[0:5]

['f__Pseudomonadaceae;g__Pseudomonas_E',
 'f__Streptococcaceae;g__Streptococcus']

In [86]:
fm_info = (
    all_info
    .filter(pl.col('taxonomy').str.contains(";f__"))
    .group_by(pl.col('taxonomy').str.replace("^([^o]__[^;]+;)+", "").str.replace(";g__.+$", ""))
    .agg(pl.col('n').sum())
    .filter(pl.col('n') > 1)
    .sort(pl.col('n'), pl.col("taxonomy"), descending = [True, False])
    .head(2)
    .with_columns(pl.lit("f").alias("rank"))
)
or_info = (
    all_info
    .filter(pl.col('taxonomy').str.contains(";o__"))
    .group_by(pl.col('taxonomy').str.replace("^([^c]__[^;]+;)+", "").str.replace(";f__.+$", ""))
    .agg(pl.col('n').sum())
    .filter(pl.col('n') > 1)
    .sort(pl.col('n'), pl.col("taxonomy"), descending = [True, False])
    .head(2)
    .with_columns(pl.lit("o").alias("rank"))
)
cl_info = (
    all_info
    .filter(pl.col('taxonomy').str.contains(";c__"))
    .group_by(pl.col('taxonomy').str.replace("^([^p]__[^;]+;)+", "").str.replace(";o__.+$", ""))
    .agg(pl.col('n').sum())
    .filter(pl.col('n') > 1)
    .sort(pl.col('n'), pl.col("taxonomy"), descending = [True, False])
    .head(2)
    .with_columns(pl.lit("c").alias("rank"))
)
ph_info = (
    all_info
    .filter(pl.col('taxonomy').str.contains(";p__"))
    .group_by(pl.col('taxonomy').str.replace("^([^d]__[^;]+;)+", "").str.replace(";c__.+$", ""))
    .agg(pl.col('n').sum())
    .filter(pl.col('n') > 1)
    .sort(pl.col('n'), pl.col("taxonomy"), descending = [True, False])
    .head(2)
    .with_columns(pl.lit("p").alias("rank"))
)
tax_info = pl.concat(
    [gn_info,
     fm_info,
     or_info,
     cl_info,
     ph_info]
)

In [88]:
with pl.Config(fmt_str_lengths=1000, tbl_width_chars=1000):
    print(tax_info)

shape: (10, 3)
┌────────────────────────────────────────────┬──────┬──────┐
│ taxonomy                                   ┆ n    ┆ rank │
│ ---                                        ┆ ---  ┆ ---  │
│ str                                        ┆ i32  ┆ str  │
╞════════════════════════════════════════════╪══════╪══════╡
│ f__Pseudomonadaceae;g__Pseudomonas_E       ┆ 120  ┆ g    │
│ f__Streptococcaceae;g__Streptococcus       ┆ 69   ┆ g    │
│ o__Burkholderiales;f__Burkholderiaceae     ┆ 183  ┆ f    │
│ o__Enterobacterales;f__Enterobacteriaceae  ┆ 155  ┆ f    │
│ c__Gammaproteobacteria;o__Enterobacterales ┆ 308  ┆ o    │
│ c__Gammaproteobacteria;o__Pseudomonadales  ┆ 262  ┆ o    │
│ p__Proteobacteria;c__Gammaproteobacteria   ┆ 971  ┆ c    │
│ p__Firmicutes;c__Bacilli                   ┆ 448  ┆ c    │
│ d__Bacteria;p__Proteobacteria              ┆ 1403 ┆ p    │
│ d__Bacteria;p__Actinobacteriota            ┆ 488  ┆ p    │
└────────────────────────────────────────────┴──────┴──────┘
